In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from PIL import Image
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

In [2]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection'

In [3]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [4]:
class SegmentationDataset(Dataset):
    def __init__(self, root, image_folder="images", mask_folder="masks", transforms=None):
        self.root = root
        self.transforms = transforms
        self.image_folder = os.path.join(root, image_folder)
        self.mask_folder = os.path.join(root, mask_folder)
        self.image_names = sorted(os.listdir(self.image_folder))
        self.mask_names = sorted(os.listdir(self.mask_folder))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_folder, self.image_names[idx])
        mask_path = os.path.join(self.mask_folder, self.mask_names[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale mask
        if self.transforms:
            image = self.transforms(image)
            mask = transforms.ToTensor()(mask)  # Mask to tensor (0-1 range)
        return {"image": image, "mask": mask}

In [5]:
def iou_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-10) if union > 0 else 1.0

def f1_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    tp = np.sum(y_true * y_pred)
    fp = np.sum(y_pred) - tp
    fn = np.sum(y_true) - tp
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    return 2 * (precision * recall) / (precision + recall + 1e-10)

In [6]:
import pandas as pd

In [7]:
def evaluate_model(model, dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set to evaluation mode

    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()  # Same loss as training
    num_samples = 0
    all_matrics = []

    with torch.no_grad():  # No gradient computation
        for sample in tqdm(dataloader):
            inputs = sample["image"].to(device)
            masks = sample["mask"].to(device)
            outputs = model(inputs)["out"]  # Shape: (batch, 1, H, W)
            loss = criterion(outputs, masks)
            total_loss += loss.item() * inputs.size(0)

            # Convert logits to binary predictions
            
            preds = torch.sigmoid(outputs) > 0.5  # Threshold at 0.5
            preds_ = preds
            masks_ = masks
            preds = preds.cpu().numpy().astype(np.uint8)
            masks = masks.cpu().numpy().astype(np.uint8)

            # Compute metrics per batch
            for i in range(inputs.size(0)):
                all_matrics.append(compute_metrics(preds_[i], masks_[i]))
                # Compute IoU and F1 score
                total_iou += iou_score(masks[i], preds[i])
                total_f1 += f1_score(masks[i], preds[i])
            num_samples += inputs.size(0)

    avg_loss = total_loss / num_samples
    avg_iou = total_iou / num_samples
    avg_f1 = total_f1 / num_samples
    
    df = pd.DataFrame(all_matrics, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall","region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
    df.to_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/deeplabv3/metrics.csv", index=False)
    return avg_loss, avg_iou, avg_f1, all_matrics

In [8]:
data_dir = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test" 
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
])

In [9]:
dataset = SegmentationDataset(data_dir, transforms=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [10]:
def create_deeplab(output_channels=1):
    model = models.segmentation.deeplabv3_resnet101(pretrained=True)
    model.classifier = DeepLabHead(2048, output_channels)
    return model

In [11]:
model = create_deeplab(output_channels=1)

model.load_state_dict(torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/deeplabv3/deeplab_rooftop_Dict_50.pth", map_location=torch.device("cuda")))
model

/home/dhruv/Documents/my_python_envs/solar_env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dhruv/Documents/my_python_envs/solar_env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [12]:
avg_loss, avg_iou, avg_f1, matrixs_array = evaluate_model(model, dataloader)

100%|██████████| 87/87 [01:04<00:00,  1.35it/s]


In [13]:
!

In [14]:
matrixs_array

[[np.float64(0.9666041890418117),
  0.983018539701952,
  0.9846391677856445,
  0.9805323321870629,
  0.9855173871683754,
  0.9666041890418117,
  0.983018539701952,
  0.9805323321870629,
  0.9855173871683754,
  1.0],
 [np.float64(0.967989528752116),
  0.9837344300972061,
  0.978053092956543,
  0.9915295653078351,
  0.9760609052759139,
  0.967989528752116,
  0.9837344300972061,
  0.9915295653078351,
  0.9760609052759139,
  1.0],
 [np.float64(0.9501284794804636),
  0.974426546227958,
  0.9884872436523438,
  0.960204073112282,
  0.9890766782780717,
  0.9501284794804636,
  0.974426546227958,
  0.960204073112282,
  0.9890766782780717,
  1.0],
 [np.float64(0.9566414801214603),
  0.9778403349213224,
  0.9976577758789062,
  0.9773112578003823,
  0.9783699851948146,
  0.9566414801214603,
  0.9778403349213224,
  0.9773112578003823,
  0.9783699851948146,
  1.0],
 [np.float64(0.0), 0.0, 0.9968948364257812, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 [np.float64(0.9279952257480347),
  0.9626530329067446,
 

In [17]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/deeplabv3/metrics.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.885407
pixel_dice                 0.923173
pixel_accuracy             0.983416
pixel_precision            0.959521
pixel_recall               0.917334
region_iou                 0.885407
region_dice                0.923173
region_precision           0.959521
region_recall              0.917334
region_success_accuracy    0.945245
dtype: float64

In [18]:
avg_loss, avg_iou, avg_f1

(0.04785289008825244,
 np.float64(0.8867130603177784),
 np.float64(0.9245704372060477))